In [ ]:
import einops
import numpy as np
import torch

from miscope import load_family
from miscope.analysis.library.fourier import get_fourier_basis

family = load_family("modulo_addition_1layer")
PRIME, SEED, DATA_SEED = 109, 485, 598
variant = family.get_variant(prime=PRIME, seed=SEED, data_seed=DATA_SEED)
checkpoint_list = variant.get_available_checkpoints()
input_set = family.generate_analysis_dataset(variant.params)

pinned_checkpoint = 1500
pinned_checkpoint_index = checkpoint_list.index(pinned_checkpoint)
device = "cuda" # TODO: Convert to device check

variant, len(checkpoint_list)

In [ ]:
# Load weights
weights = variant.artifacts.load_epoch("parameter_snapshot", pinned_checkpoint)
W_E = weights["W_E"]        # (d_vocab, d_model)
W_U = weights["W_U"]        # (d_model, d_vocab)
W_Q = weights["W_Q"]        # (n_heads, d_model, d_head)
W_K = weights["W_K"]        # (n_heads, d_model, d_head)
W_O = weights["W_O"]        # (n_heads, d_head, d_model)
W_V = weights["W_V"]        # (n_heads, d_model, d_head)
W_in = weights["W_in"]      # (d_model, d_mlp)
W_out = weights["W_out"]    # (d_mlp, d_model)
#W_QK_Circuit = W_Q @ W_K
W_OV_Circuit = W_V @ W_O
W_neurons = W_E @ W_V @ W_O @ W_in
W_logits = W_out @ W_U

head_count = W_Q.shape[0]
W_QKT = []
W_OV = []
QK_circuit_per_head = []
OV_circuit_per_head = []

for head in range(head_count):
    W_Q_h, W_K_h = W_Q[head, :], W_K[head, :]

    w_qkt_h = W_Q_h @ W_K_h.T       # (d_model, d_head) @ (d_head, d_model)
    w_ov_h =  W_V[head] @ W_O[head] # (d_model, d_head) @ (d_head, d_model)

    W_QKT.append(w_qkt_h) 
    W_OV.append(w_ov_h)

    #print(f"w_qkt_h.shape: {w_qkt_h.shape}")
    #print(f"w_ov_h.shape: {w_ov_h.shape}")

    QK_circuit_per_head.append(W_E @ W_QKT[head] @ W_E.T)    # (d_vocab, d_model) @ (d_model, d_model) @ (d_model, d_vocab)
    OV_circuit_per_head.append(W_U.T @ W_OV[head] @ W_E.T)   # (d_vocab, d_model) @ (d_model, d_model) @ (d_model, d_vocab)

print("----- Architecture-defined objects -----")
print(f"W_E.shape: {W_E.shape}\n W_U.shape: {W_U.shape}")
print(f"W_Q.shape: {W_Q.shape}\n W_K.shape: {W_K.shape}")
print(f"W_O.shape: {W_O.shape}\n W_V.shape: {W_V.shape}")
print(f"W_in.shape: {W_in.shape}\n W_out.shape: {W_out.shape}")

print(f"Head Count: {head_count}")

print("----- Derived objects -----")
print(f"W_OV_Circuit.shape: {W_OV_Circuit.shape}")
print(f"W_neurons.shape: {W_neurons.shape}\n W_logits.shape: {W_logits.shape}")

print(f"QK_circuit[0].shape: {QK_circuit_per_head[0].shape}")
print(f"OV_circuit[0].shape: {OV_circuit_per_head[0].shape}")


In [ ]:
# Fourier Basis Objects - Weights
fourier_basis, fourier_basis_names = get_fourier_basis(PRIME)
fourier_basis_ndarray = fourier_basis.cpu().detach().numpy()

W_E_no_equals_token = W_E[:-1]
W_E_Fourier = fourier_basis_ndarray @ W_E_no_equals_token
W_E_Fourier_norms = np.linalg.norm(W_E_Fourier, axis=-1)

W_U_Fourier = W_U @ fourier_basis_ndarray.T
W_U_Fourier_norms = np.linalg.norm(W_U_Fourier, axis=0)

print("----- Derived objects: Fourier Basis -----")
print(f"W_E_Fourier.shape: {W_E_Fourier.shape}")
print(f"W_E_Fourier_norms.shape: {W_E_Fourier_norms.shape}")
print(f"W_U_Fourier.shape: {W_U_Fourier.shape}")
print(f"W_U_Fourier_norms.shape: {W_U_Fourier_norms.shape}")

In [ ]:
# Activations
probe_logits, probe_cache = variant.run_with_cache(input_set, pinned_checkpoint)

act_embed = probe_cache["embed.hook_out"]
#act_unembed = probe_cache["unembed.hook_out"]

act_attn_out = probe_cache["blocks.0.attn.hook_out"]
act_attn_pattern = probe_cache["blocks.0.attn.hook_pattern"]
act_mlp_out = probe_cache["blocks.0.mlp.hook_out"]
act_resid_post = probe_cache["blocks.0.hook_out"]

print("----- Architecture-defined activation objects -----")
print(f"act_embed.shape: {act_embed.shape}, device: {act_embed.device}")
#print(f"act_unembed.shape: {act_unembed.shape}")
print(f"act_attn_out.shape: {act_attn_out.shape}")
print(f"act_attn_pattern.shape: {act_attn_pattern.shape}")
print(f"act_mlp_out.shape: {act_mlp_out.shape}")
print(f"act_resid_post.shape: {act_resid_post.shape}")

$$
\text{neuron\_frequency\_norm}[k,n] \;=\; \frac{\displaystyle\sum_{x,y \,\in\, \{0,\,c_k,\,s_k\}} \hat{A}_n[x,y]^2}{\displaystyle\sum_{i,j} \hat{A}_n[i,j]^2}
$$


In [ ]:
# Analysis - Neuron clusters

# ---------------------------------------------------------------------------
# neuron_frequency_norm[k, n]:
#   Fraction of neuron n's centered activation power carried by frequency k.
#
#   act_mlp_fourier_basis[n, i, j] is the 2-D Fourier transform of neuron n's
#   activation over the (a, b) input grid. Index layout along each axis:
#       0 -> DC (constant);  cos_k at 2k-1;  sin_k at 2k   (k 1-indexed)
#
#                  sum over x,y in {0, cos_k, sin_k} of  A_hat[n, x, y]^2
#   value  =  ---------------------------------------------------------------
#                       sum over all i,j of  A_hat[n, i, j]^2
#
#   Numerator = the 3x3 "pure frequency-k" block:
#       (cos_k,cos_k) (cos_k,sin_k) (sin_k,cos_k) (sin_k,sin_k)  -> freq k on both axes
#       (0,cos_k) (0,sin_k) (cos_k,0) (sin_k,0)                  -> freq k on one axis, const on other
#       (0,0)                                                    -> DC/DC corner
#   Denominator = neuron's total Fourier energy. Each row is thus a per-neuron
#   distribution over frequencies (cross-frequency leakage ~ 0 once grokked).
#
# DC/DC double-count note:
#   The cos_k/sin_k indices are disjoint across frequencies, so the ONLY cell
#   shared by every frequency block is (0,0). The loop therefore re-visits (0,0)
#   once per frequency (counted P//2 times). This is harmless ONLY because
#   `act_mlp_fourier_basis[:, 0, 0] = 0.0` zeros it first (mean-centering): the
#   value added each time is 0. Do NOT remove that line as "just centering" —
#   it is what prevents the neuron's mean from leaking into every frequency.
#   The off-axis DC terms (0,cos_k)/(cos_k,0) are unique to one frequency and
#   are genuinely distinct coefficients, so counting both is correct.
# ---------------------------------------------------------------------------


act_mlp_out_nobatch = act_mlp_out[:, -1, :]
act_mlp_out_nobatch_ndarray = act_mlp_out_nobatch.cpu().detach().numpy()

act_mlp_fourier_basis = (
    fourier_basis_ndarray
    @ einops.rearrange(act_mlp_out_nobatch_ndarray, "(a b) neuron -> neuron a b", a=PRIME, b=PRIME)
    @ fourier_basis_ndarray.T
)

print(f"act_mlp_fourier_basis.shape: {act_mlp_fourier_basis.shape}")

# Center these by removing the mean - doesn't matter!
act_mlp_fourier_basis[:, 0, 0] = 0.0
neuron_frequency_norm = np.zeros([PRIME // 2, act_mlp_fourier_basis.shape[0]])

for frequency in range(0, PRIME // 2):
    cos_index = 2 * (frequency + 1) - 1
    sin_index = 2 * (frequency + 1)
    for x in [0, cos_index, sin_index]:
        for y in [0, cos_index, sin_index]:
            print(x, y, act_mlp_fourier_basis[:, x, y].shape)
            neuron_frequency_norm[frequency] += act_mlp_fourier_basis[:, x, y] ** 2

neuron_frequency_norm = neuron_frequency_norm / np.power(act_mlp_fourier_basis, 2).sum(axis=(-1, -2))[None, :]            

In [12]:
# Per-neuron frequency concentration: fraction of each neuron's centered
# Fourier power that sits in frequency k's 3x3 block {DC, cos_k, sin_k}^2.
A = act_mlp_fourier_basis                       # (n_neurons, P, P)
A[:, 0, 0] = 0.0                                # drop DC/DC (mean); see note above
total_power = (A ** 2).sum(axis=(-2, -1))       # (n_neurons,)

n_freqs = PRIME // 2
neuron_frequency_norm = np.zeros((n_freqs, A.shape[0]))
for k in range(n_freqs):
    idx = [0, 2 * k + 1, 2 * k + 2]             # {DC, cos_k, sin_k}
    block = A[:, idx][:, :, idx]                # (n_neurons, 3, 3)
    neuron_frequency_norm[k] = (block ** 2).sum(axis=(-2, -1)) / total_power
